# Unlike-Charge Collision Bounce Demo

This notebook demonstrates a head-on pair with charges `+1` and `-1` in 2D. The pair is integrated with `RegularizedIntegrator(...; backend = :lifted_pair)` and `collision_bounce_radius`, so close approaches use the lifted pair backend and the geometric bounce provides the explicit pass-through continuation near `r = 0`.

The bounce is intended for `L = 0` head-on cases. It is not a generic cure for nonzero-angular-momentum collisions.

In [ ]:
using WeberElectrodynamics
using LinearAlgebra: norm
using Printf: @printf

## Setup

The two particles start at rest on the x-axis. With unlike charges, they accelerate toward each other, pass through the bounce radius, and repeat as a bounded head-on oscillation in this explicitly reflected model.

In [ ]:
m = 1.0
qmag = 1.0
c = 100.0
r0 = 0.25
dt = 5e-5
bounce_r = 0.006

sys = HamiltonianSystem(2, 2)
prob = HamiltonianProblem(
    sys,
    (0.0, 1.0),
    [r0 / 2, 0.0, -r0 / 2, 0.0],
    zeros(4);
    masses = [m, m],
    charges = [qmag, -qmag],
    c = c,
    dt = dt,
    maximum_iterations = 200,
)

alg = RegularizedIntegrator(
    SymmetricProjectionIntegrator();
    backend = :lifted_pair,
    r_on = 0.03,
    r_off = 0.05,
    max_substeps = 4096,
    collision_bounce_radius = bounce_r,
    warn_on_fallback = false,
)

## Solve

In [ ]:
sol = solve(prob, alg; save_stride = 10)
diag = sol.regularization

@printf "retcode              = %s\n" sol.retcode
@printf "saved points          = %d\n" length(sol.t)
@printf "used backend          = %s\n" diag.used_backend
@printf "lifted pair steps     = %d\n" diag.lifted_pair_steps
@printf "total substeps        = %d\n" diag.total_substeps
@printf "min encounter distance = %.6e\n" diag.min_encounter_distance

## Diagnostics

In [ ]:
separation(q) = sqrt((q[1] - q[3])^2 + (q[2] - q[4])^2)
rs = separation.(sol.q)
x1 = getindex.(sol.q, 1)
x2 = getindex.(sol.q, 3)

energy = compute_energy_timeseries(sol)
relative_energy_error =
    (energy.total_energy .- energy.total_energy[1]) ./ abs(energy.total_energy[1]) .* 100
n_close_passes = count(k -> rs[k] < rs[k-1] && rs[k] < rs[k+1], 2:(length(rs)-1))

@printf "min separation        = %.6e\n" minimum(rs)
@printf "max separation        = %.6e\n" maximum(rs)
@printf "close-pass minima     = %d\n" n_close_passes
@printf "max energy drift      = %.6e %%\n" maximum(abs, relative_energy_error)

## Optional Plots

Run this cell if `Plots.jl` is available in the active Julia environment.

In [ ]:
using Plots

p_sep = plot(sol.t, rs; xlabel = "t", ylabel = "r12", label = "separation")
hline!(p_sep, [bounce_r]; label = "bounce radius", linestyle = :dash)

p_x = plot(sol.t, x1; xlabel = "t", ylabel = "x", label = "particle 1")
plot!(p_x, sol.t, x2; label = "particle 2")

p_e = plot(
    energy.t,
    relative_energy_error;
    xlabel = "t",
    ylabel = "energy drift (%)",
    label = "relative drift",
)

plot(p_sep, p_x, p_e; layout = (3, 1), size = (900, 800))